# File 02/7

### DESCRIPTION:
- Filters tense forms based on morphological tags. Simple and compound verb forms can be identified and filtered.

#### NOTE: 
- There is a mismatch between the predefined morphology schema (all possible values) and the empirical data (only a subset is attested). 
- For **compound tenses**, this can be explained by the fact that auxiliary verb(s) and main verb are annotated separately.
- Thus the user has to find the main verb and its corresponding auxiliary verb(s), e.g.:

##### Verbs that can be found with a simple lookup like "--a-------" in column "Morphology"
```
AORIST
  Structure:
    Aux1: —
    Aux2: —
    Main: aorist
  Example:
    stavich''
```
##### Compound tense verbs that can be found by examining 
- a) the present form of auxiliary verb 'byti' ("--p-------") plus
- b) the resultative form of the main verb ("--s-------")

```
PERFECT
  Structure:
    Aux1: present (byti)
    Aux2: —
    Main: resultative
  Example:
    jesm' (po)stavil''
```

- WARNING:

At the moment

      - FILE: 6_Verb_Verb_Combos.ipynb
      - FUNCTION: mark_compound_verbs
 defines forms of "byti" as the only valid auxiliary verbs for tenses
(cf.: AUX_LEMMAS = {"быти"}). 
If other verbs are considered valid auxiliary verbs for compound tenses, add them to the set AUX_LEMMAS, e.g.: 

AUX_LEMMAS = {"быти", "имѣти"}

#### INPUT:
- Annotated tokens with morphological feature strings (fixed-position encoding)

#### OUTPUT:
- Subset of tokens matching specified tense criteria

In [1]:
# standard library imports 
import ast
# third-party imports
import pandas as pd 

In [2]:
# schema of the 10-character positional morphology encoding
# (cf. index notation) in the "Morphology" column
morphology = {
    "person": { # 0 
        '1': 'first person',
        '2': 'second person',
        '3': 'third person',
        'x': 'uncertain person'
    },
    "number": { # 1
        's': 'singular',
        'd': 'dual',
        'p': 'plural',
        'x': 'uncertain number'
    },
    "tense": { # 2
        'p': 'present',
        'i': 'imperfect',
        'r': 'perfect',
        's': 'resultative',
        'a': 'aorist',
        'u': 'past',
        'l': 'pluperfect',
        'f': 'future',
        't': 'future perfect',
        'x': 'uncertain tense'
    },
    "mood": { # 3 
        'i': 'indicative',
        's': 'subjunctive',
        'm': 'imperative',
        'o': 'optative',
        'n': 'infinitive',
        'p': 'participle',
        'd': 'gerund',
        'g': 'gerundive',
        'u': 'supine',
        'x': 'uncertain mood',
        'y': 'finiteness unspecified',
        'e': 'indicative or subjunctive',
        'f': 'indicative or imperative',
        'h': 'subjunctive or imperative',
        't': 'finite'
    },
    "voice": { # 4
        'a': 'active',
        'm': 'middle',
        'p': 'passive',
        'e': 'middle or passive',
        'x': 'unspecified'
    },
    "gender": { # 5
        'm': 'masculine',
        'f': 'feminine',
        'n': 'neuter',
        'p': 'masculine or feminine',
        'o': 'masculine or neuter',
        'r': 'feminine or neuter',
        'q': 'masculine, feminine or neuter',
        'x': 'uncertain gender'
    },
    "case": { # 6 
        'n': 'nominative',
        'a': 'accusative',
        'o': 'oblique',
        'g': 'genitive',
        'c': 'genitive or dative',
        'e': 'accusative or dative',
        'd': 'dative',
        'b': 'ablative',
        'i': 'instrumental',
        'l': 'locative',
        'v': 'vocative',
        'x': 'uncertain case',
        'z': 'no case'
    },
    "degree": { # 7
        'p': 'positive',
        'c': 'comparative',
        's': 'superlative',
        'x': 'uncertain degree',
        'z': 'no degree'
    },
    "strength": { # 8 
        'w': 'weak',
        's': 'strong',
        't': 'weak or strong'
    },
    "inflection": { # 9 
        'n': 'non-inflecting',
        'i': 'inflecting'
    }
}

In [3]:
# read input file 
df = pd.read_csv("OUTPUTS/dataframe_02_6.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string", 
                     "Sentence ID": "Int64"
                 }
)

def drop_unnamed_columns(df):
    """
    Drop columns whose names start with 'Unnamed', typically created
    when saving/loading DataFrames with an index in CSV format.
    """
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

df = drop_unnamed_columns(df)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_63140/1640784324.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_6.csv",


In [4]:
# create sub dataframe consisting of only verbs

df_verbs = df[
    # keep only verbs 
    (df['POS'] == 'V-') 
    # remove auxiliary verbs 
    # as main verbs already cointain info on its corresponding aux verbs
    & (df["Is_Compound_Aux"]==False)]
# amount of verb forms 
verb_count = len(df_verbs)
verb_count

41525

In [5]:
# file type of the columns "col" is simple string 
# to access elements, convert the columns in list "col" to an actual list 
cols = [
    "Compound_Aux_Forms",
    "Compound_Aux_IDs",
    "Compound_Aux_Lemmas",
    "Compound_Aux_Morphology"
]

for col in cols:
    df_verbs.loc[:, col] = df_verbs[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [7]:
# assert that all columns starting with "Compound..." contain values of type "list" 
col_Compound_Aux_Negation = "Compound_Aux_Negation"
for col in df_verbs:
    if col.startswith("Compound") and col != col_Compound_Aux_Negation:
        col_type = df_verbs[col].apply(type).value_counts()
        #print(col_type)
        assert (df_verbs[col].apply(lambda x: isinstance(x, list))).all()

In [8]:
# make sure all Compound_Aux verbs have been removed
assert not df_verbs['Is_Compound_Aux'].any()

### CREATE QUERY
IMPORTANT: Auxiliary verbs are position-sensitive. The order in AUX_LEMMAS determines how AUX_MORPH is applied.
Reordering AUX_LEMMAS changes which morphological constraints apply to which auxiliary.

In [9]:
FEATURE_INDEX = {
    "person": 0,
    "number": 1,
    "tense": 2,
    "mood": 3,
    "voice": 4,
    "gender": 5,
    "case": 6,
    "degree": 7,
    "strength": 8,
    "inflection": 9
}

def has_feature(morph, feature, value):
    idx = FEATURE_INDEX[feature]
    return isinstance(morph, str) and len(morph) > idx and morph[idx] == value


def match_morph(morph, constraints):
    return all(has_feature(morph, f, v) for f, v in constraints.items())


def find_forms(df, AUX_LEMMAS=None, AUX_FORMS=None, AUX_MORPH=None, MAIN_MORPH=None):

    def match(row):
        x_lem   = row["Compound_Aux_Lemmas"]
        x_forms = row["Compound_Aux_Forms"]
        x_morph = row["Compound_Aux_Morphology"]

        y_morph  = row["Morphology"]

        # --- basic sanity ---
        if not (isinstance(x_lem, list) and isinstance(x_morph, list)):
            return False

        # --- length consistency ---
        n = len(x_lem)
        if AUX_LEMMAS and len(AUX_LEMMAS) != n:
            return False
        if AUX_FORMS and len(AUX_FORMS) != n:
            return False
        if AUX_MORPH and len(AUX_MORPH) != n:
            return False

        # --- lemmas ---
        if AUX_LEMMAS:
            if x_lem != AUX_LEMMAS:
                return False

        # --- forms ---
        if AUX_FORMS:
            if not (isinstance(x_forms, list) and x_forms == AUX_FORMS):
                return False

        # --- morphology (positional dependency)  ---
        if AUX_MORPH:
            for m, constraints in zip(x_morph, AUX_MORPH):
                if not match_morph(m, constraints):
                    return False


        # ---- MAIN VERB ----
        if MAIN_MORPH: 
            if not match_morph(y_morph, MAIN_MORPH): 
                return False
        
        return True

    return df[df.apply(match, axis=1)]


### EXAMPLE USE CASES for filtering values:

In [10]:
# Example 1a: here "ѥси былъ шелъ" (jesi byl'' shel'') 
# AND "суть б҃ы не правили вѣры"  (sut' by ne pravili)
# main verb: resultative 
# auxiliary verb 1: present tense 
# auxiliary verb 2: ANYTHING (but None)
byti_present_byti_ANYTHING_main_resultative = find_forms(
    df_verbs,
    MAIN_MORPH=
        {"tense":"s"} # main - resultative
    , 
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
    AUX_MORPH=[
        {"tense":"p"}, # aux1 - present tense 
        {} # aux2 - ANYTHING  
    ]
)

In [11]:
byti_present_byti_ANYTHING_main_resultative

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation
29114,lav,6494: Vladimir consults with representatives o...,orv,126590,1724781,правили,правити,правити,False,False,...,не,Nizhny_Novgorod,суть б҃ы не правили вѣры,True,False,"[суть, б҃ы]","[1724778, 1724780]","[быти, быти]","[3ppia----i, 3saia----i]","[False, False]"
227498,suz-lav,6685,orv,212117,2295936,шелъ,ити,ити,False,False,...,NaN,Moscow,ѥси былъ шелъ,True,False,"[ѥси, былъ]","[2295937, 2295938]","[быти, быти]","[2spia----i, -sspamn-si]","[False, False]"


In [12]:
# Example 1b: here "ѥси былъ шелъ" (jesi byl'' shel'') 
# main verb: resultative 
# auxiliary verb 1: present tense 
# auxiliary verb 2: resultative
byti_present_byti_resultative_main_resultative = find_forms(
    df_verbs,
    MAIN_MORPH=
        {"tense":"s"} # main - resultative
    , 
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
    AUX_MORPH=[
        {"tense":"p"}, # aux1 - present tense 
        {"tense":"s"} # aux2 - resultative 
    ]
)

In [13]:
byti_present_byti_resultative_main_resultative

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation
227498,suz-lav,6685,orv,212117,2295936,шелъ,ити,ити,False,False,...,NaN,Moscow,ѥси былъ шелъ,True,False,"[ѥси, былъ]","[2295937, 2295938]","[быти, быти]","[2spia----i, -sspamn-si]","[False, False]"


In [14]:
# Example 2: 
# aux1: byti in present; second person singular
# MAIN verb: ANYTHING (not delimited)
byti_present = find_forms(
    df_verbs,
    AUX_LEMMAS=["быти"], # aux1 
    AUX_MORPH=[
        {"tense": "p", "person": "2", "number": "s"}, # aux1 
    ]
)

In [15]:
byti_present.head(2)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation
487,birchbark,109,orv,210149,2287509,кѹпилъ,купити,купити,False,False,...,NaN,Novgorod,еси кѹпилъ робѹ,True,False,[еси],[2287510],[быти],[2spia----i],[False]
537,birchbark,109,orv,210162,2287557,възалъ,възяти,в_зяти,False,False,...,не,Novgorod,тꙑ еси не възалъ кѹнъ (техъ),True,False,[еси],[2287555],[быти],[2spia----i],[False]


In [16]:
byti_byti_ANYTHING = find_forms(
    df_verbs,
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
)

In [17]:
byti_byti_ANYTHING.head(2)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation
3266,pskov,Untitled,orv,207300,1794635,изволили,изволити,изволити,False,False,...,NaN,Moscow,"вы бы есте изволили воли (две, мои)",True,False,"[бы, есте]","[1794630, 1794631]","[быти, быти]","[3saia----i, 2ppia----i]","[False, False]"
3280,pskov,Untitled,orv,139964,1794648,сняли,съняти,с_няти,False,False,...,NaN,Moscow,бы есте сняли колокол (вечной),True,False,"[бы, есте]","[1794646, 1794647]","[быти, быти]","[3saia----i, 2ppia----i]","[False, False]"


# Store tenses in data frame 

### Example 1: store tense "perfect" in newly created df_verbs["TENSE"]

In [18]:
mask_perfect = find_forms(
    df_verbs,
    MAIN_MORPH={"tense":"s"},
    AUX_LEMMAS=["быти"],
    AUX_MORPH=[{"tense": "p"}]
).index

In [19]:
df_verbs = df_verbs.copy()
df_verbs.loc[mask_perfect, "TENSE"] = "perfect"

In [20]:
df_verbs.loc[df_verbs["TENSE"] == "perfect"].head(2)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation,TENSE
11,mst,Mstislav’s letter,orv,189407,2157784,повелѣлъ,повелѣти,повелети,False,False,...,Novgorod,"азъ ѥсмь повелѣлъ ѿдати бѹицѣ (и (съ, съ, съ))",True,False,[ѥсмь],[2157785],[быти],[1spia----i],[False],perfect
116,mst,Mstislav’s letter,orv,189417,2157888,далъ,дати,дати,False,False,...,Novgorod,"ꙗ ѥсмь далъ блюдо (серебрьно, въ)",True,False,[ѥсмь],[2157889],[быти],[1spia----i],[False],perfect


### Example 2: store tense "pluperfect I" in newly created df_verbs["TENSE"]

In [21]:
mask_pluperfect_I = find_forms(
    df_verbs, 
    MAIN_MORPH={"tense":"s"},
    AUX_LEMMAS=["быти"],
    AUX_MORPH=[{"tense": "a"}]
).index

In [22]:
df_verbs.loc[mask_pluperfect_I, "TENSE"] = "pluperfect_I"

In [23]:
df_verbs.loc[df_verbs["TENSE"] == "pluperfect_I"].head(2)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology,Compound_Aux_Negation,TENSE
1349,birchbark,705,orv,210732,2290635,посолале,посълати,пос_лати,False,False,...,Novgorod,бꙑхо посолале,True,False,[бꙑхо],[2290634],[быти],[1saia----i],[False],pluperfect_I
1536,birchbark,752,orv,210750,2290816,притькль,притещи,притещи,False,False,...,Novgorod,бꙑ притькль,True,False,[бꙑ],[2290813],[быти],[2saia----i],[False],pluperfect_I
